In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Rotate)=
# Rotate

*Rotating atomic coordinates by rotation matrices or transformations.*

The function {func}`molsysmt.structure.rotate` applies proper 3D rotations to the atomic coordinates of a molecular system using rotation matrices or spatial transformation objects.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.structure.rotate`.
:::


## Basic usage

Let's show how to rotate Met-enkephalin by 90 degrees around the $z$-axis using a $3\times 3$ rotation matrix:


In [2]:
import molsysmt as msm
import numpy as np


In [3]:
molsys = msm.convert(msm.systems['Met-enkephalin']['met_enkephalin.h5msm'])
molsys = msm.structure.center(molsys, selection='all')


:::{note}
:class: dropdown

**Demo Systems Catalog**

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::


We inspect the initial 3D orientation of the peptide:


In [4]:
molsysviewer_htmlfile = '_static/views/tools_structure_rotate_1.html'


In [5]:
msm.view(molsys)


'<iframe src="../../../../_static/views/tools_structure_rotate_1.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

We define an active 90-degree counterclockwise rotation matrix around the $z$-axis ($R_z(\pi/2)$) and apply it using {func}`molsysmt.structure.rotate`:


In [6]:
rotation_matrix = [
    [0.0, -1.0, 0.0],
    [1.0,  0.0, 0.0],
    [0.0,  0.0, 1.0]
]
molsys_rotated = msm.structure.rotate(molsys, rotation=rotation_matrix)


Visualizing the rotated system in 3D confirms the new spatial orientation:


In [7]:
molsysviewer_htmlfile = '_static/views/tools_structure_rotate_2.html'


In [8]:
msm.view(molsys_rotated)


'<iframe src="../../../../_static/views/tools_structure_rotate_2.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

## Rotating around a specific center

By default, rotation is performed around the coordinate origin `[0, 0, 0]`. Setting `rotation_center` allows rotating coordinates around a specific anchor point (e.g. the centroid of a residue or backbone atoms):


In [9]:
ca_center = msm.structure.get_center(molsys, selection='atom_name=="CA"')
molsys_ca_rot = msm.structure.rotate(molsys, rotation=rotation_matrix, rotation_center=ca_center)


## Rotating using SciPy Rotation objects

{func}`molsysmt.structure.rotate` natively accepts SciPy `Rotation` instances (such as Euler angles, rotation vectors, or quaternions):


In [10]:
from scipy.spatial.transform import Rotation

rot_euler = Rotation.from_euler('y', 45, degrees=True)
molsys_euler = msm.structure.rotate(molsys, rotation=rot_euler)


(Tutorial_Rotate_axis_angle)=
## Constructing from an axis and angle

You can choose an orientation using one unit-free axis and an explicit degree or
radian angle. Positive 90 degrees about +z takes +x to +y. The constructor returns
a dimensionless `(3, 3)` matrix for the active row-coordinate convention
`coordinates @ rotation_matrix.T`. A zero axis or bare angle has no supported
interpretation. Axis magnitude does not change the rotation; reversing its sign
reverses the positive angle direction.

The example checks the matrix and the resulting coordinates while preserving
the original molecular system. It uses the same centered Met-enkephalin as above.


In [11]:
axis_angle_rotation = msm.structure.get_rotation_matrix_from_axis_angle(
    axis=[0, 0, 2], angle='90 degrees')
np.testing.assert_allclose(
    np.array([1.0, 0.0, 0.0]) @ axis_angle_rotation.T, [0.0, 1.0, 0.0],
    rtol=0, atol=1e-12)
before_axis_angle = msm.pyunitwizard.get_value(
    msm.get(molsys, coordinates=True), to_unit='nm').copy()
molsys_rotated = msm.structure.rotate(molsys, rotation=axis_angle_rotation)
after_axis_angle = msm.pyunitwizard.get_value(
    msm.get(molsys_rotated, coordinates=True), to_unit='nm')
np.testing.assert_allclose(
    after_axis_angle, before_axis_angle @ axis_angle_rotation.T, rtol=0, atol=1e-12)
np.testing.assert_array_equal(
    msm.pyunitwizard.get_value(msm.get(molsys, coordinates=True), to_unit='nm'),
    before_axis_angle)


The angle is converted through PyUnitWizard without changing your unit policy.
The resulting matrix also works with `rotation_center` and selected atoms or
structures. Rotation changes coordinates; boxes and periodic reconstruction
remain separate.

:::{admonition} API documentation
:class: dropdown

- {func}`molsysmt.structure.get_rotation_matrix_from_axis_angle`
- {func}`molsysmt.structure.rotate`
:::


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Convert <Tutorial_Convert>`: Convert molecular systems between different forms with {func}`molsysmt.basic.convert`.
- {ref}`Center <Tutorial_Center>`: Center a molecular system around coordinate origin or reference points with {func}`molsysmt.structure.center`.
- {ref}`View <Tutorial_View>`: Render molecular systems interactively with {func}`molsysmt.basic.view`.
- {ref}`Get center <Tutorial_Get_center>`: Calculate centroid positions with {func}`molsysmt.structure.get_center`.
- {ref}`Translate <Tutorial_Translate>`: Translate molecular coordinates by a displacement vector with {func}`molsysmt.structure.translate`.
- {ref}`Flip <Tutorial_Flip>`: Reflect coordinates across a plane with {func}`molsysmt.structure.flip`.
- {ref}`Align principal axes <Tutorial_Align_principal_axes>`: Orient principal inertia axes along target Cartesian directions with {func}`molsysmt.structure.align_principal_axes`.
:::
